# ANÁLISE | The Numbers Dataset
___

In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/processed/tabela_tratada.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2700 entries, 0 to 2699
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   genre                 2699 non-null   object 
 1   day                   2700 non-null   int64  
 2   month                 2700 non-null   int64  
 3   year                  2700 non-null   int64  
 4   nome                  2700 non-null   object 
 5   tn_legs               2694 non-null   float64
 6   domestic_share        2697 non-null   float64
 7   domestic_bo           2697 non-null   float64
 8   domestic_bo_original  2697 non-null   float64
 9   rating                2667 non-null   object 
 10  tempo                 2650 non-null   float64
 11  producao              2696 non-null   object 
 12  estreia_bilheteria    2682 non-null   float64
 13  semana_2              2684 non-null   float64
 14  semana_3              2666 non-null   float64
 15  semana_4             

In [4]:
div_size = 50

num_generos = df['genre'].nunique()
num_ratings = df['rating'].nunique()
num_prods = df['producao'].nunique()
num_movies = df.shape[0]
num_years = df['year'].nunique()
years_min = df['year'].min()
years_max = df['year'].max()

print('-' * div_size)
print('THE NUMBERS DATASET | Informações Gerais')
print('-' * div_size)
print(f'Número de filmes:               {num_movies}')
print(f'Número de gêneros:              {num_generos}')
print(f'Número de classificações:       {num_ratings}')
print(f'Número de tipos de produção:    {num_prods}')
print(f'Número de anos:                 {num_years} ({years_min}-{years_max})')
print('-' * div_size)
print('ÚLTIMA EXTRAÇÃO: 09/11/2026')
print('-' * div_size)

--------------------------------------------------
THE NUMBERS DATASET | Informações Gerais
--------------------------------------------------
Número de filmes:               2700
Número de gêneros:              14
Número de classificações:       5
Número de tipos de produção:    6
Número de anos:                 27 (2000-2026)
--------------------------------------------------
ÚLTIMA EXTRAÇÃO: 09/11/2026
--------------------------------------------------


In [5]:
df_generos = df['genre'].value_counts()

fig = px.bar(
    df_generos,
    title='Distribuição de filmes por gênero',
    orientation='h'
)

fig.show()

In [6]:
df_ratings = df['rating'].value_counts().reset_index()
df_ratings.columns = ['rating', 'quantidade']

fig = px.pie(
    df_ratings,
    names='rating',
    values='quantidade',
    title='Distribuição de filmes por classificação indicativa',
    hole=0.4
)

fig.show()

In [7]:
df_prod = df['producao'].value_counts().reset_index()
df_prod.columns = ['producao', 'quantidade']

fig = px.pie(
    df_prod,
    names='producao',
    values='quantidade',
    title='Distribuição de filmes por tipo de produção',
    hole=0.4
)

fig.show()

In [8]:
numerico = ['tn_legs', 'domestic_share', 'domestic_bo', 'tempo', 'estreia_bilheteria', 'semana_2', 'semana_3', 'semana_4']
df_numerico = df[numerico].dropna()
matriz_corr = df_numerico.corr()

# 4. Cria o mapa de calor interativo com Plotly Express
fig = px.imshow(
    matriz_corr,
    text_auto='.2f',          # Mostra os valores numéricos com 2 casas decimais dentro dos quadrados
    color_continuous_scale='RdBu_r', # Escala de cores (Vermelho a Azul divergente)
    title='Mapa de Calor: Correlação entre Métricas de Bilheteria'
)

fig.update_layout(
    xaxis_title='Variáveis',
    yaxis_title='Variáveis',
    width=700,
    height=600
)

# Exibe o gráfico no navegador ou IDE compatível
fig.show()

In [24]:
fig = px.scatter(
    df,
    y='tn_legs',          # Eixo X: Multiplicador de "pernas" (sustentabilidade do filme)
    x='domestic_bo',      # Eixo Y: Bilheteria doméstica ajustada pela inflação
    hover_name='nome',    # Exibe o nome do filme ao passar o mouse
    hover_data=['year', 'rating', 'estreia_bilheteria'], # Dados extras no tooltip
    color='genre',        # Opcional: Colorir os pontos por gênero (se a coluna existir)
    title='Relação entre Multiplicador de Pernas (TN Legs) e Bilheteria Doméstica Ajustada',
    labels={
        'tn_legs': 'TN Legs (Multiplicador)',
        'domestic_bo': 'Bilheteria Doméstica Ajustada ($)',
        'genre': 'Gênero'
    },
    opacity=0.75
)

# 3. Ajustes visuais (Opcional, mas altamente recomendado para bilheterias)
fig.update_layout(
    yaxis_title='TN Legs',
    xaxis_title='Bilheteria Doméstica Ajustada (Escala Log)',
    #yaxis_type='log',     # Transforma o eixo Y em escala logarítmica para melhor visualização
    width=900,
    height=600,
    template='plotly_white'
)

# Exibe o gráfico interativo
fig.show()

In [18]:
# 2. Agrupa por ano e calcula a média da bilheteria doméstica ajustada
# (Certifique-se de que a coluna 'year' existe no seu DataFrame)
df_anual = df.groupby('year')['domestic_bo'].mean().reset_index()

# Opcional: Se quiser remover eventuais anos nulos ou 0 que possam corromper a linha
df_anual = df_anual.dropna()

# 3. Cria o gráfico de linhas com Plotly Express
fig = px.line(
    df_anual,
    x='year',                  # Eixo X: Anos
    y='domestic_bo',           # Eixo Y: Arrecadação média
    markers=True,              # Adiciona pontos visíveis em cada ano (facilita o hover)
    title='Evolução da Arrecadação Média Doméstica por Ano (Ajustada pela Inflação)',
    labels={
        'year': 'Ano de Lançamento',
        'domestic_bo': 'Média de Bilheteria Doméstica ($)'
    }
)

# 4. Ajustes visuais para formatar o layout e os eixos
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Arrecadação Média Ajustada ($)',
    width=1000,
    height=600,
    template='plotly_white',
    hovermode='x unified'      # Une o tooltip de hover para facilitar a leitura por ano
)

# Formata o eixo Y para exibir os valores de forma amigável (opcional)
fig.update_yaxes(tickprefix='$', separatethousands=True)

# Exibe o gráfico interativo
fig.show()

In [11]:
analise = df[['estreia_bilheteria', 'tn_legs', 'domestic_bo']].dropna()

# Calcula a matriz de correlação de Pearson
correlacao = analise.corr()

print("Matriz de Correlação:")
print(correlacao['domestic_bo'])

Matriz de Correlação:
estreia_bilheteria    0.903292
tn_legs               0.028099
domestic_bo           1.000000
Name: domestic_bo, dtype: float64


In [12]:
df_plot = df.dropna(subset=['estreia_bilheteria', 'domestic_bo']).copy()

# Cria o gráfico de dispersão
fig = px.scatter(
    df_plot,
    x='estreia_bilheteria',
    y='domestic_bo',
    hover_name='nome',
    #color='genre',
    hover_data=['year', 'genre', 'tn_legs'],
    trendline='ols',          # Linha de tendência estatística
    title='Relação Direta: Bilheteria de Estreia vs. Bilheteria Doméstica Total',
    labels={
        'estreia_bilheteria': 'Bilheteria de Estreia ($)',
        'domestic_bo': 'Bilheteria Doméstica Total ($)'
    },
    opacity=0.6
)

# Ajustes vitais: Escala logarítmica em AMBOS os eixos para dados de bilheteria
fig.update_layout(
    #xaxis_type='log',
    #yaxis_type='log',
    xaxis_title='Bilheteria de Estreia',
    yaxis_title='Bilheteria Doméstica Total',
    width=900,
    height=600,
    template='plotly_white'
)

fig.show()

In [16]:
df_plot = df.dropna(subset=['semana_2', 'domestic_bo']).copy()

# Cria o gráfico de dispersão
fig = px.scatter(
    df_plot,
    x='semana_2',
    y='domestic_bo',
    hover_name='nome',
    #color='genre',
    hover_data=['year', 'genre', 'tn_legs'],
    trendline='ols',          # Linha de tendência estatística
    title='Relação Direta: Segunda Semana vs. Bilheteria Doméstica Total',
    labels={
        'semana_2': 'Segunda Semana ($)',
        'domestic_bo': 'Bilheteria Doméstica Total ($)'
    },
    opacity=0.6
)

# Ajustes vitais: Escala logarítmica em AMBOS os eixos para dados de bilheteria
fig.update_layout(
    #xaxis_type='log',
    #yaxis_type='log',
    xaxis_title='Segunda Semana',
    yaxis_title='Bilheteria Doméstica Total',
    width=900,
    height=600,
    template='plotly_white'
)

fig.show()

In [21]:
df_plot = df.dropna(subset=['tempo', 'domestic_bo']).copy()

# Cria o gráfico de dispersão
fig = px.scatter(
    df_plot,
    x='tempo',
    y='domestic_bo',
    hover_name='nome',
    #color='genre',
    hover_data=['year', 'genre', 'tn_legs'],
    trendline='ols',          # Linha de tendência estatística
    title='Relação Direta: Tempo vs. Bilheteria Doméstica Total',
    labels={
        'tempo': 'Tempo',
        'domestic_bo': 'Bilheteria Doméstica Total ($)'
    },
    opacity=0.6
)

# Ajustes vitais: Escala logarítmica em AMBOS os eixos para dados de bilheteria
fig.update_layout(
    #xaxis_type='log',
    #yaxis_type='log',
    xaxis_title='Tempo',
    yaxis_title='Bilheteria Doméstica Total',
    width=900,
    height=600,
    template='plotly_white'
)

fig.show()

In [22]:
df_plot = df.dropna(subset=['rating', 'domestic_bo']).copy()

# Opcional: ordenar os ratings de forma lógica (se desejar)
# ordem_ratings = ['G', 'PG', 'PG-13', 'R', 'NC-17']

# 2. Cria o Box Plot
fig = px.box(
    df_plot,
    x='rating',
    y='domestic_bo',
    color='rating',           # Colore cada categoria de rating com uma cor diferente
    hover_name='nome',        # Mostra o nome do filme ao passar o mouse nos outliers
    hover_data=['year', 'genre', 'estreia_bilheteria'],
    title='Distribuição da Bilheteria Doméstica Ajustada por MPA Rating',
    labels={
        'rating': 'MPA Rating (Classificação Etária)',
        'domestic_bo': 'Bilheteria Doméstica Ajustada ($)'
    }
)

# 3. Ajustes visuais (Escala logarítmica no eixo Y é obrigatória para bilheteria)
fig.update_layout(
    yaxis_type='log',
    yaxis_title='Bilheteria Doméstica Ajustada ($ - Escala Log)',
    xaxis_title='Classificação Etária (Rating)',
    width=900,
    height=600,
    template='plotly_white',
    showlegend=False          # Esconde a legenda lateral já que o eixo X já identifica as cores
)

# Exibe o gráfico interativo
fig.show()